# 01: Explore Volve production and fit an Arps baseline

Goal: build the decline-curve baseline that the ML models in later notebooks must beat.

Data: `data/Volve production data.xlsx` (see `data/README.md`).

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import numpy as np

from volve import arps, data, metrics

## Load the daily data and list the wells

In [ ]:
df = data.load_daily('../data/Volve production data.xlsx')
df.groupby(['NPD_WELL_BORE_NAME', 'WELL_TYPE'])['BORE_OIL_VOL'].sum().sort_values(ascending=False)

## Pick a producer and average to monthly rates

15/9-F-12 is Volve's main producer. Monthly averaging removes shut-in days, which decline-curve analysis does not model.

In [ ]:
WELL = '15/9-F-12'
well = data.producer(df, WELL)
q = data.monthly_oil_rate(well)

fig, ax = plt.subplots(figsize=(10, 4))
q.plot(ax=ax)
ax.set_ylabel('Oil rate (Sm3/d)')
ax.set_title(f'{WELL}: monthly average oil rate')

## Fit Arps on the first 80% of the history and test on the last 20%

Start the fit at peak rate, where decline begins.

In [ ]:
q_decline = q.loc[q.idxmax():]
t = np.arange(len(q_decline))  # months since peak
split = int(0.8 * len(t))

params = arps.fit(t[:split], q_decline.values[:split])
forecast = arps.predict(t[split:], params)
actual = q_decline.values[split:]

print(params)
print(f'Test RMSE: {metrics.rmse(actual, forecast):.1f} Sm3/d')
print(f'Test MAPE: {metrics.mape(actual, forecast):.1f} %')
print(f'Cumulative volume error: {metrics.cumulative_error_pct(actual, forecast):+.1f} %')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(q_decline.index, q_decline.values, label='Actual')
ax.plot(q_decline.index[:split], arps.predict(t[:split], params), label='Arps fit (train)')
ax.plot(q_decline.index[split:], forecast, '--', label='Arps forecast (test)')
ax.axvline(q_decline.index[split], color='grey', lw=0.8)
ax.set_ylabel('Oil rate (Sm3/d)')
ax.legend()

## Notes for the next notebook

- Rate changes from choke adjustments and water breakthrough break Arps' assumptions. Record where the baseline fails.
- Next: an ML model using lagged rates plus operating variables (on-stream hours, choke size, wellhead and downhole pressure), scored on the same test window.